# 14 · Summarize points in polygons against shapely

**Question.** Does Fieldwork's `summarize_polygons` widget assign the right locations to the
right polygons, and does it account for them honestly?

This is the John Snow example's actual question: how many deaths fall nearest each pump.
Fieldwork decides membership with its own point-in-polygon test; the reference uses GEOS
predicates.

**The boundary policy is the whole difficulty.** Fieldwork's `include` means strictly inside
*or on the boundary*, which in GEOS terms is `covers`, not `contains`. `exclude` is
`contains`. Getting that mapping wrong would give a check that passes on data with no
boundary cases and fails the moment one appears — so the mapping is declared, and the number
of locations lying on an edge is reported whether or not any exist.

Criteria fixed before running: every per-polygon count, numeric total and missing-value count
exact, and the summary exact. Counts are integers; there is no tolerance to choose.

In [ ]:
# Only what this notebook imports.
NEEDED = ["shapely"]
try:
    import pyodide_js
    await pyodide_js.loadPackage(NEEDED)
    print("Requested from the Pyodide distribution:", ", ".join(NEEDED))
except ImportError:
    print("Not a Pyodide kernel; using the ambient environment.")
except Exception as error:
    print("loadPackage failed:", type(error).__name__, error)
    try:
        import piplite
        await piplite.install(NEEDED)
    except Exception as fallback:
        print("piplite also failed:", type(fallback).__name__, fallback)

## Parity

In [ ]:
import json, sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))
from checks.check_14_summarize_polygons import run

report = run(Path.cwd() / "fixtures" / "summarize-polygons-soho-001.json")
print("criterion:", report["criterion"])
print()
print("agrees:", report["agrees"], "over", report["points"], "locations in", report["polygons"], "polygons")
print()
print(f"{'polygon':<12} {'count':>7} {'shapely':>8} {'total':>8} {'shapely':>8}")
for row in report["perPolygon"]:
    print(f"{row['id']:<12} {row['fieldworkCount']:>7} {row['referenceCount']:>8} "
          f"{row['fieldworkTotal']:>8} {row['referenceTotal']:>8}")
print()
print("summary, fieldwork:", {k: v for k, v in report["fieldwork"].items() if k != "valueField"})
print("summary, shapely:  ", {k: v for k, v in report["reference"].items() if k != "valueField"})

## What this fixture does not exercise

**No location in this data lies on a polygon edge, and none belongs to more than one
polygon.** So the `covers`-versus-`contains` distinction — the part of this widget most
likely to be got wrong — is declared and *untested* by the parity check above.

That is worth saying plainly rather than letting a green result imply otherwise. The cell
below constructs the case the fixture lacks.

In [ ]:
from shapely.geometry import Point, Polygon

# Two cells meeting along x = 0, and a location exactly on the shared edge.
left = Polygon([(-1, -1), (0, -1), (0, 1), (-1, 1)])
right = Polygon([(0, -1), (1, -1), (1, 1), (0, 1)])
on_edge = Point(0, 0)
inside_left = Point(-0.5, 0)

for label, geos, predicate in [("include", "covers", lambda p, q: p.covers(q)),
                               ("exclude", "contains", lambda p, q: p.contains(q))]:
    hits = [name for name, cell in (("left", left), ("right", right)) if predicate(cell, on_edge)]
    print(f"boundary={label:<8} (GEOS {geos:<8}): the edge location belongs to {hits or ['nothing']}")

print()
print("So under 'include' a single location is counted twice -- once for each cell -- and under")
print("'exclude' it is counted not at all and reported as unmatched. Both are defensible and")
print("they are different numbers, which is why the policy is a declared parameter and travels")
print("in the fixture rather than being assumed.")
print()
print(f"A control: a location well inside the left cell belongs to "
      f"{[n for n, c in (('left', left), ('right', right)) if c.covers(inside_left)]} either way.")

## Dive deeper · why the per-polygon totals must not be added

Every membership is retained. A location inside two polygons counts for **both**, so the
per-polygon counts and totals are not parts of a whole and adding them double-counts.

For *this* data the warning does not bite — nothing overlaps — and the sum happens to be
correct. That is the dangerous case: a reader who adds the column here gets the right answer
and learns the wrong habit.

In [ ]:
fixture = json.loads((Path.cwd() / "fixtures" / "summarize-polygons-soho-001.json").read_text())
counts = sum(row["fieldworkCount"] for row in report["perPolygon"])
totals = sum(row["fieldworkTotal"] for row in report["perPolygon"])
summary = report["fieldwork"]

print(f"locations, summed over polygons   {counts}")
print(f"locations in the layer            {summary['records']}")
print(f"deaths, summed over polygons      {totals}")
print()
print(f"multiply-assigned locations        {summary['multiple']}")
print(f"locations on a polygon edge        {report['locationsOnAPolygonEdge']}")
print()
if summary["multiple"] == 0:
    print("Nothing overlaps here, so the sums are right -- by luck of the data, not by rule.")
    print("With one overlapping location the summed count would exceed the layer's record count")
    print("and the summed deaths would exceed the real total, with no error raised anywhere.")
print()
print(report["notEstablished"])

### Reading the result

`agrees` means Fieldwork and GEOS assigned the same locations to the same polygons and
accounted for the same values. It does not mean nearest-pump assignment models which pump
anybody drank from: it assumes the nearest pump by straight-line distance, which is what the
widget computes and not what anybody did.